# 두 Agent 기본 구현 · Python / LLM / ML·DL 확장
5쪽 통합계획서에 맞춘 시작용 소스입니다. **가상 A기업 합성 자료**로 실행합니다.

- 기본 실행은 외부 API·Neon·웹 서버 없이 가능합니다.
- 산업: 검수된 주장·근거 재사용, 명시된 비율의 변화 계산, 경쟁 항목의 부족 자료 표시.
- 기술: 입력 검사, 이동평균·수익률·거래량 변화·변동성 계산.
- LLM: 오프라인 문구와 실제 Solar 호출을 명확히 구분합니다.
- ML·DL: 연결 위치와 미학습 도구 예시만 제공하며 모델이 학습됐다고 표시하지 않습니다.


In [ ]:
# 필요한 경우에만 현재 커널에 설치
# %pip install "pandas>=2,<3" "python-dotenv>=1,<2"
# 실제 Solar 사용 시에만 추가 설치:
# %pip install "langchain-upstage>=0.6" "langchain>=0.3"


In [ ]:
from pathlib import Path
import sys
import json
from copy import deepcopy
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "agent/basic_agents.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("압축 해제한 프로젝트의 notebooks 폴더에서 열어주세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from agent.basic_agents import (
    AgentSettings, DemoLLM, SolarLLM, IndustryBusinessAgent,
    TechnicalAnalysisAgent, compare_results,
)
from agent.research_contract import validate_report
report = json.loads((ROOT / "examples/data/a_company_report_v1.json").read_text("utf-8-sig"))
validate_report(report)
original = deepcopy(report)
settings = AgentSettings(window=3, literacy="L2", style="U")
print("자료:", report["subject"]["name"], report["dataMode"], "내용 버전", report["contentRevision"])


## 1. Python 분석부터 실행
LLM을 지정하지 않으면 계산·근거 연결만 수행합니다. 상태 partial은 경쟁 근거·실제 시세 검증 등 남은 항목을 뜻합니다.
이 초기 산업 Agent는 자유 문서를 자동 분석하지 않습니다. 검수된 주장과 명시된 수치에서 시작합니다.


## 실행 전 기업별 품질 검사
기간·결측·최신성·원문 연결을 검사합니다. 합성 자료는 실제 최신성 통과로 표시하지 않습니다.
실행 가능은 현재 규칙 통과이며 사실 검증이나 승인 완료가 아닙니다. 일부 가능은 한계를 유지해 진행하고 보류는 Agent·LLM·모델 실행을 막습니다.


In [ ]:
from agent.quality_gate import assess_report, gate_table, run_checked
QUALITY_REFERENCE_DATE = "2026-10-27"  # 재현 가능한 검사 기준일. 실제 분석일로 수정
MAX_PRICE_AGE_DAYS = 5  # 달력일 경과 정책. 공식 휴장일 검증과 별도
MAX_DOC_AGE_DAYS = 400  # 문서 경과 경고 기준, 공시 의무주기 판정 아님
gates, quality_checks = assess_report(
    report, window=settings.window, reference_date=QUALITY_REFERENCE_DATE,
    max_price_age_days=MAX_PRICE_AGE_DAYS, max_doc_age_days=MAX_DOC_AGE_DAYS)
display(pd.DataFrame(gate_table(gates)))
display(pd.DataFrame(quality_checks))


In [ ]:
industry = run_checked(IndustryBusinessAgent(settings=settings), report, gates["industry_business_model"])
technical = run_checked(TechnicalAnalysisAgent(settings=settings), report, gates["technical_analysis"])

def show_result(result):
    display(pd.DataFrame([{key: result[key] for key in
        ("agentType", "status", "reportId", "contentRevision", "dataMode", "explanationStatus")}]))
    print("주장")
    display(pd.DataFrame(result["claims"]))
    print("계산")
    display(pd.DataFrame(result["calculations"]))
    print("근거")
    display(pd.DataFrame(result["evidence"]))
    print("한계:", *result["limitations"], sep="\n- ")
    print("다음 요청:", *result["followUpRequests"], sep="\n- ")
    if result["explanation"]:
        print("설명:", result["explanation"])

show_result(industry)
display(pd.DataFrame(industry["competitionTable"]))
show_result(technical)


## 2. 오프라인 설명 연결
DemoLLM은 고정 템플릿입니다. 실제 생성형 모델의 출력이나 성능으로 제시하지 마세요.
설명은 독립 필드이며 Python 수치·주장·근거를 덮어쓰지 않습니다.


In [ ]:
industry_demo = run_checked(IndustryBusinessAgent(DemoLLM(), settings), report, gates["industry_business_model"])
technical_demo = run_checked(TechnicalAnalysisAgent(DemoLLM(), settings), report, gates["technical_analysis"])
print(industry_demo["explanation"])
print(technical_demo["explanation"])
comparison = compare_results(industry_demo, technical_demo)
print(json.dumps(comparison, ensure_ascii=False, indent=2))


## 3. 실제 LLM 연결 · 기본값 OFF
API 키는 프로젝트 `.env`의 UPSTAGE_API_KEY에, 모델은 SOLAR_MODEL에 설정합니다.
키를 셀·출력·Git에 넣지 마세요. 이 셀을 켜면 입력 자료가 Upstage로 전송되고 API 비용이 발생할 수 있습니다.
기존 설정을 바꾸면 커널을 다시 시작하세요. LLM 응답은 unverified_llm으로 표시되며 별도 Verifier 검사가 필요합니다.


In [ ]:
RUN_LIVE_LLM = False
if RUN_LIVE_LLM:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env", override=False)
    solar = SolarLLM()
    live_industry = run_checked(IndustryBusinessAgent(solar, settings), report, gates["industry_business_model"])
    live_technical = run_checked(TechnicalAnalysisAgent(solar, settings), report, gates["technical_analysis"])
    show_result(live_industry)
    show_result(live_technical)
else:
    print("실제 LLM 미호출: Python 분석과 오프라인 데모만 실행했습니다.")


## 4. ML·DL 연결 위치
두 Agent 생성자의 model_tool에 predict(payload) 도구를 전달합니다.
반드시 modelVersion, trainingCutoff, status, output, limitations를 반환합니다.

| 관점 | ML 기준선 | DL 후보 | 평가 |
|---|---|---|---|
| 산업 문단 분류 | TF-IDF + 로지스틱 회귀 | 한국어 사전학습 분류기 | 문서·기업·시점 분리 / Macro F1 |
| 기술 변동성 추정 | 최근 변동성 유지·회귀 | 작은 LSTM | 시간순 분리·정답구간 중복 방지 / MAE |

산업 분류기의 입력 문단과 정답 라벨 수집은 후속 작업입니다. 모델 출력은 근거 검증이나 매수 판단이 아닙니다.
아래는 **미학습 연결 예시**이므로 실제 예측을 생성하지 않습니다.


In [ ]:
class NotTrainedModelHook:
    def predict(self, payload):
        # 여기에 검증된 모델 로드 → 특징 생성 → predict → 버전/한계 반환을 구현합니다.
        return {
            "modelVersion": "not-trained",
            "trainingCutoff": None,
            "status": "not_run",
            "output": None,
            "limitations": ["학습·평가 후 연결할 자리입니다. 실제 예측 없음."],
        }

extended = run_checked(TechnicalAnalysisAgent(DemoLLM(), settings, model_tool=NotTrainedModelHook()), report, gates["technical_analysis"])
display(pd.DataFrame(extended["modelResults"]))


## 5. 보류·원본 보존 검사
합성 가격 101·99·97의 이동평균은 99입니다. 데이터 길이가 설정에 못 미치면 계산을 보류합니다.


In [ ]:
assert industry["calculations"][0]["result"] == 10.0  # 30% → 40% = 10%p
assert technical["calculations"][0]["result"] == 99
short = TechnicalAnalysisAgent(settings=AgentSettings(window=20)).run(report)
assert short["status"] == "withheld" and short["calculations"] == []
assert report == original
print("계산·기간 부족·입력 원본 보존 검사 통과")


## 6. 선택적 로컬 저장
파일은 파생 Agent 결과입니다. 원본 보고서의 claim/version을 자동 갱신하거나 사람의 승인으로 취급하지 않습니다.
실제 자료 연결 시 report/evidence 계약을 맞추고, 시세는 ISO 날짜·자료 기준시점·조정 메타데이터가 필요합니다.
공식 달력·기업행위·공개시각의 검증과 Neon 조회는 별도 도구에서 처리합니다.


In [ ]:
SAVE_RESULTS = False
if SAVE_RESULTS:
    out = ROOT / "private_outputs/basic_agents"
    out.mkdir(parents=True, exist_ok=True)
    for name, result in [("industry", industry_demo), ("technical", technical_demo)]:
        (out / f"{name}.json").write_text(
            json.dumps(result, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    print("로컬 결과 저장 완료")


## 보류 시 실제 실행 차단 확인
관측 5개에 window=20을 요구해 기술 Agent를 보류시킵니다. LLM이 지정돼 있어도 설명·모델·계산을 실행하지 않습니다.


In [ ]:
blocked_settings = AgentSettings(window=20)
blocked_gates, _ = assess_report(report, window=20, reference_date=QUALITY_REFERENCE_DATE)
blocked_result = run_checked(TechnicalAnalysisAgent(DemoLLM(), blocked_settings), report, blocked_gates["technical_analysis"])
display(pd.DataFrame(gate_table(blocked_gates)))
assert blocked_result["status"] == "withheld"
assert blocked_result["explanationStatus"] == "not_run"
assert blocked_result["calculations"] == []
print("보류 검사 통과: 계산·LLM 실행 없이 사유만 반환")
